In [1]:
from plntxps import *
import pandas as pd
import lmfit
import re
from lmfitxps import models

data processed with plnt_xps version 0.1.1
our crew is replaceable, your package isn't!


In [8]:
def format_satellite_name(name):
    formatted_name = re.sub(r" ", "_", name)
    formatted_name = re.sub(r",", "", formatted_name)
    return formatted_name

satellites = pd.read_csv('example data/perkin elmer mg satellites.csv',
                sep = '\t').to_dict(orient='index')

for satellite in satellites.values():
    satellite['name'] = format_satellite_name(satellite['name'])

In [9]:
satellites

{0: {'name': 'alpha_1_2', 'position': 0.0, 'intensity': 0.0},
 1: {'name': 'alpha_3', 'position': 8.4, 'intensity': 8.0},
 2: {'name': 'alpha_4', 'position': 10.1, 'intensity': 4.1},
 3: {'name': 'alpha_5', 'position': 17.6, 'intensity': 0.6},
 4: {'name': 'alpha_6', 'position': 20.3, 'intensity': 0.5},
 5: {'name': 'beta', 'position': 48.7, 'intensity': 0.5}}

In [10]:
peak = models.ConvGaussianDoniachSinglett(prefix = "example_", independent_vars = ["x"])

In [11]:
peak.make_params()

name,value,initial value,min,max,vary,expression
example_amplitude,100.000000,None,0.00000000,inf,True,
example_sigma,0.20000000,None,0.00000000,inf,True,
example_gamma,0.02000000,None,-inf,inf,True,
example_gaussian_sigma,0.20000000,None,0.00000000,inf,True,
example_center,100.000000,None,0.00000000,inf,True,
example_gaussian_fwhm,0.47096000,None,-inf,inf,False,2*example_gaussian_sigma*1.1774
example_lorentzian_fwhm,0.41005962,None,-inf,inf,False,example_sigma*(2+example_gamma*2.5135+(example_gamma*3.6398)**4)


In [12]:
peak.set_param_hint("amplitude", value = 50, min = 0)

In [24]:
parent_prefix = 'example_'

satellite_models = []
for satellite in satellites.values():
    full_prefix = parent_prefix + satellite['name']
    model = models.ConvGaussianDoniachSinglett(prefix = full_prefix, independent_vars = ["x"])

    model.set_param_hint('amplitude',
        expr = f"{parent_prefix}amplitude*{satellite['intensity']}/100")
    model.set_param_hint('sigma',
        expr = f"{parent_prefix}sigma")
    model.set_param_hint('gamma', 
        expr = f"{parent_prefix}gamma")
    model.set_param_hint('gaussian_sigma',
        expr = f"{parent_prefix}gaussian_sigma")
    model.set_param_hint('center',
        expr = f"{parent_prefix}center+{satellite['position']}")
    
    g_fwhm_expr = f'2*{full_prefix}gaussian_sigma*1.1774'
    model.set_param_hint('gaussian_fwhm', expr=g_fwhm_expr)
    l_fwhm_expr = f'{full_prefix}sigma*(2+{full_prefix}gamma*2.5135+({full_prefix}gamma*3.6398)**4)'
    model.set_param_hint('lorentzian_fwhm', expr=l_fwhm_expr)
    
    satellite_models.append(model)

In [25]:
satellite_models[0].make_params()

NameError: name 'example_amplitude' is not defined

NameError: name 'example_amplitude' is not defined